# Simple Linear Regression

**What this notebook covers:**
- Understanding the regression line ŷ = β₀ + β₁x
- Implementing Simple Linear Regression from scratch using OLS formulas
- Comparing with scikit-learn's LinearRegression
- Residual analysis and assumption checking
- Hyperparameter sensitivity experiments

**Prerequisites:**
- Introduction to Supervised Learning
- Introduction to Regression Algorithms
- Basic statistics: mean, variance, covariance, correlation
- Python, NumPy, Pandas basics

---

**Dataset:** House Prices — Advanced Regression Techniques  
🔗 [Kaggle Dataset Link](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)  
*Credits: Kaggle / Dean De Cock — Ames Housing dataset. We use GrLivArea (above-ground living area in sq ft) as the single predictor for SalePrice, making this a true Simple Linear Regression problem.*

In [ ]:
# --- All Imports ---
import numpy as np                        # Numerical operations and OLS math
import pandas as pd                       # Data loading and manipulation
import matplotlib.pyplot as plt           # Plotting regression lines and residuals
import seaborn as sns                     # Statistical visualizations
from sklearn.linear_model import LinearRegression   # Sklearn SLR implementation
from sklearn.model_selection import train_test_split, cross_val_score  # Splitting and CV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # Evaluation
import warnings
warnings.filterwarnings('ignore')         # Suppress minor warnings for clean output

np.random.seed(42)  # Reproducibility for all random operations

## Part 1: Theory Recap

Five key things to remember about Simple Linear Regression:

- **SLR fits a straight line ŷ = β₀ + β₁x**: β₁ is the slope (change in y per unit x) and β₀ is the intercept (y value when x = 0). Just two numbers describe the entire model.
- **OLS finds the best line by minimizing squared residuals**: β₁ = Cov(x,y)/Var(x) and β₀ = ȳ - β₁x̄. These are closed-form formulas — no iteration needed.
- **Residuals = actual - predicted**: A good model has residuals that are random (no pattern), centered at zero, and roughly normally distributed.
- **Assumptions matter (LINE)**: Linearity, Independence, Normality of residuals, Equal variance. Violating them does not crash the model but makes results unreliable.
- **R² measures fit, not causation**: High R² means the line explains the variance well — it does NOT mean x causes y or that the model will generalize perfectly.

## Loading the Dataset

We use the **Ames Housing dataset** and focus on a single feature to keep this truly "Simple" Linear Regression:
- **Input feature (x):** `GrLivArea` — above-ground living area in square feet
- **Target variable (y):** `SalePrice` — house sale price in USD

Living area is the strongest single predictor of house price — a perfect candidate for SLR.

In [ ]:
# Load the Ames Housing dataset from a local source
df = pd.read_csv("train.csv")

# Select only the columns we need for SLR
df_slr = df[['Gr Liv Area', 'SalePrice']].dropna()

print("Dataset Shape:", df_slr.shape)
print("\n--- First 5 Rows ---")
display(df_slr.head())

print("\n--- Statistical Summary ---")
display(df_slr.describe())

# Quick check: correlation between our feature and target
corr = df_slr['Gr Liv Area'].corr(df_slr['SalePrice'])
print(f"\nCorrelation between GrLivArea and SalePrice: {corr:.4f}")
print("Strong positive correlation — good candidate for SLR!" if corr > 0.5 else "Weak correlation — SLR may not perform well.")

# Scatter plot to visually confirm linear relationship
plt.figure(figsize=(8, 5))
plt.scatter(df_slr['Gr Liv Area'], df_slr['SalePrice'], alpha=0.4, color='steelblue', edgecolors='white')
plt.title('GrLivArea vs SalePrice — Checking Linearity', fontsize=13, fontweight='bold')
plt.xlabel('Above Ground Living Area (sq ft)', fontsize=11)
plt.ylabel('Sale Price ($)', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Preprocessing

For Simple Linear Regression, preprocessing is minimal since we have one clean numerical feature. We need to:
1. **Remove outliers** — a few extremely large houses distort the line significantly
2. **Split into train/test** — 80/20 split for honest evaluation
3. **Keep original scale** — we intentionally avoid scaling here so β₁ stays interpretable in real units ($ per sq ft)

In [ ]:
# Step 1: Remove extreme outliers — houses > 4000 sq ft are rare and distort the line
# INTERVIEW NOTE: Outliers have disproportionate leverage on OLS — one point can shift β₁ dramatically
df_clean = df_slr[df_slr['Gr Liv Area'] < 4000].copy()
print(f"Rows before outlier removal: {len(df_slr)}")
print(f"Rows after  outlier removal: {len(df_clean)}")

# Step 2: Extract feature and target as arrays
x = df_clean['Gr Liv Area'].values   # Single feature — 1D array
y = df_clean['SalePrice'].values     # Target — 1D array

# Step 3: Train/test split — 80% train, 20% test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(f"\nTraining samples : {len(x_train)}")
print(f"Test samples     : {len(x_test)}")
print(f"\nFeature range (train): {x_train.min():.0f} — {x_train.max():.0f} sq ft")
print(f"Target range  (train): ${y_train.min():,.0f} — ${y_train.max():,.0f}")
print(f"\nBaseline RMSE (predict mean always): ${np.std(y_test):,.0f}")

## Part 2: From Scratch Implementation

We implement Simple Linear Regression using the **Ordinary Least Squares (OLS)** closed-form formulas:

```
β₁ = Cov(x, y) / Var(x)  =  Σ(xᵢ - x̄)(yᵢ - ȳ) / Σ(xᵢ - x̄)²
β₀ = ȳ - β₁ × x̄
```

These formulas come from taking the derivative of the MSE loss with respect to β₀ and β₁, setting both to zero, and solving. No iteration — just two equations, two unknowns.

In [ ]:
class SimpleLinearRegressionScratch:
    """
    Simple Linear Regression from scratch using OLS closed-form solution.
    Fits: ŷ = β₀ + β₁x
    """

    def __init__(self):
        self.beta_0 = None   # Intercept
        self.beta_1 = None   # Slope

    def fit(self, x, y):
        # Step 1: Compute means of x and y
        x_mean = np.mean(x)
        y_mean = np.mean(y)

        # Step 2: Compute β₁ = Cov(x,y) / Var(x)
        # Numerator: covariance — how x and y move together
        numerator   = np.sum((x - x_mean) * (y - y_mean))
        # Denominator: variance of x — how much x varies on its own
        denominator = np.sum((x - x_mean) ** 2)
        # INTERVIEW NOTE: If denominator = 0, all x values are identical — no information in x
        self.beta_1 = numerator / denominator

        # Step 3: Compute β₀ = ȳ - β₁ × x̄
        # This ensures the line always passes through the centroid (x̄, ȳ)
        self.beta_0 = y_mean - self.beta_1 * x_mean

    def predict(self, x):
        # Apply learned line equation: ŷ = β₀ + β₁x
        return self.beta_0 + self.beta_1 * x

    def score(self, x, y):
        # Compute R² manually
        y_pred  = self.predict(x)
        ss_res  = np.sum((y - y_pred) ** 2)       # Unexplained variance
        ss_tot  = np.sum((y - np.mean(y)) ** 2)   # Total variance
        return 1 - (ss_res / ss_tot)

# Train the model
slr_scratch = SimpleLinearRegressionScratch()
slr_scratch.fit(x_train, y_train)

print("=== Simple Linear Regression — From Scratch ===")
print(f"β₁ (Slope)     : {slr_scratch.beta_1:.4f}")
print(f"β₀ (Intercept) : {slr_scratch.beta_0:.4f}")
print(f"\nInterpretation: For every 1 sq ft increase in living area,")
print(f"the predicted sale price increases by ${slr_scratch.beta_1:,.2f}")

In [ ]:
# Evaluate the from-scratch model on test set
y_pred_scratch = slr_scratch.predict(x_test)

mse_s  = mean_squared_error(y_test, y_pred_scratch)
rmse_s = np.sqrt(mse_s)
mae_s  = mean_absolute_error(y_test, y_pred_scratch)
r2_s   = r2_score(y_test, y_pred_scratch)

print("=== From Scratch Evaluation on Test Set ===")
print(f"MSE  : {mse_s:>20,.2f}")
print(f"RMSE : ${rmse_s:>18,.2f}  ← average prediction error")
print(f"MAE  : ${mae_s:>18,.2f}  ← median prediction error")
print(f"R²   : {r2_s:>20.4f}  ← model explains {r2_s*100:.1f}% of price variance")

# Residual analysis
residuals = y_test - y_pred_scratch
print(f"\nResidual mean  : ${residuals.mean():,.2f}  (should be ~0)")
print(f"Residual std   : ${residuals.std():,.2f}")

## Part 3: Sklearn Implementation

Scikit-learn's `LinearRegression` uses the same OLS math internally but is optimized using LAPACK routines for numerical stability. The API requires X to be 2D (reshape with `-1, 1` for a single feature).

Our from-scratch β₀ and β₁ should match sklearn's `intercept_` and `coef_` exactly — this is our correctness check.

In [ ]:
# Sklearn requires 2D input — reshape single feature from (n,) to (n, 1)
X_train_sk = x_train.reshape(-1, 1)
X_test_sk  = x_test.reshape(-1, 1)

# Train sklearn model
slr_sklearn = LinearRegression()
slr_sklearn.fit(X_train_sk, y_train)
y_pred_sklearn = slr_sklearn.predict(X_test_sk)

rmse_sk = np.sqrt(mean_squared_error(y_test, y_pred_sklearn))
mae_sk  = mean_absolute_error(y_test, y_pred_sklearn)
r2_sk   = r2_score(y_test, y_pred_sklearn)

print("=== Sklearn LinearRegression Results ===")
print(f"β₁ (coef_)      : {slr_sklearn.coef_[0]:.4f}")
print(f"β₀ (intercept_) : {slr_sklearn.intercept_:.4f}")
print(f"RMSE            : ${rmse_sk:,.2f}")
print(f"MAE             : ${mae_sk:,.2f}")
print(f"R²              : {r2_sk:.4f}")

print("\n=== Comparison: Scratch vs Sklearn ===")
print(f"β₁ match: Scratch={slr_scratch.beta_1:.4f} | Sklearn={slr_sklearn.coef_[0]:.4f} | {'✅ Match' if abs(slr_scratch.beta_1 - slr_sklearn.coef_[0]) < 0.01 else '❌ Mismatch'}")
print(f"β₀ match: Scratch={slr_scratch.beta_0:.2f} | Sklearn={slr_sklearn.intercept_:.2f} | {'✅ Match' if abs(slr_scratch.beta_0 - slr_sklearn.intercept_) < 1 else '❌ Mismatch'}")
print(f"R²  match: Scratch={r2_s:.4f}  | Sklearn={r2_sk:.4f}   | {'✅ Match' if abs(r2_s - r2_sk) < 0.001 else '❌ Mismatch'}")

In [ ]:
# --- Visualisation 1: Regression Line + Data ---
# Shows the fitted line against actual data points
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Regression line on scatter plot
x_line = np.linspace(x_train.min(), x_train.max(), 200)
y_line = slr_scratch.predict(x_line)

axes[0].scatter(x_train, y_train, alpha=0.3, color='steelblue', label='Training Data', s=20)
axes[0].scatter(x_test, y_test, alpha=0.5, color='orange', label='Test Data', s=20)
axes[0].plot(x_line, y_line, color='red', linewidth=2.5, label=f'Regression Line\nŷ = {slr_scratch.beta_0:,.0f} + {slr_scratch.beta_1:.2f}x')
axes[0].set_title('Simple Linear Regression — Best Fit Line', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Above Ground Living Area (sq ft)', fontsize=11)
axes[0].set_ylabel('Sale Price ($)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: Residual plot — checks homoscedasticity assumption
# INTERVIEW NOTE: Residuals should be randomly scattered around 0
# A fan/cone pattern means heteroscedasticity — equal variance assumption violated
residuals_test = y_test - y_pred_scratch
axes[1].scatter(y_pred_scratch, residuals_test, alpha=0.4, color='purple', s=20)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2, label='Zero Residual Line')
axes[1].set_title('Residual Plot — Checking Assumptions', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Sale Price ($)', fontsize=11)
axes[1].set_ylabel('Residual (Actual - Predicted) ($)', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Simple Linear Regression: GrLivArea → SalePrice', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Visualisation 2: Residual Distribution ---
# Checks the Normality assumption — residuals should be bell-shaped
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of residuals
axes[0].hist(residuals_test, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[0].set_title('Distribution of Residuals', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Residual Value ($)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Actual vs Predicted
axes[1].scatter(y_test, y_pred_scratch, alpha=0.4, color='green', s=20)
min_val = min(y_test.min(), y_pred_scratch.min())
max_val = max(y_test.max(), y_pred_scratch.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_title(f'Actual vs Predicted  |  R² = {r2_s:.4f}', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Actual Sale Price ($)', fontsize=11)
axes[1].set_ylabel('Predicted Sale Price ($)', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Residual Analysis — Checking LINE Assumptions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

SLR has no traditional hyperparameters (no α, no depth) — but we can experiment with two important practical factors:

1. **Outlier sensitivity** — how much do outliers shift β₁ and R²?
2. **Training size effect** — how does the amount of training data affect model performance?

These experiments build intuition about SLR's weaknesses in practice.

In [ ]:
# Experiment 1: Effect of training set size on R²
# Learning curve — how much data is enough?
train_sizes = np.linspace(0.05, 0.95, 20)  # 5% to 95% of training data
train_r2_scores = []
test_r2_scores  = []

for size in train_sizes:
    n = max(2, int(size * len(x_train)))
    # Randomly sample n points from training set
    idx = np.random.choice(len(x_train), n, replace=False)
    x_sub, y_sub = x_train[idx], y_train[idx]

    model = SimpleLinearRegressionScratch()
    model.fit(x_sub, y_sub)

    train_r2_scores.append(model.score(x_sub, y_sub))
    test_r2_scores.append(model.score(x_test, y_test))

# Experiment 2: Outlier sensitivity — inject extreme outliers and observe β₁ shift
outlier_counts = [0, 1, 3, 5, 10, 20]
beta1_values   = []
r2_values      = []

for n_outliers in outlier_counts:
    x_exp = x_train.copy().astype(float)
    y_exp = y_train.copy().astype(float)
    if n_outliers > 0:
        # Add extreme outliers: very large houses with very low prices
        outlier_x = np.full(n_outliers, 5000.0)
        outlier_y = np.full(n_outliers, 50000.0)
        x_exp = np.concatenate([x_exp, outlier_x])
        y_exp = np.concatenate([y_exp, outlier_y])
    model = SimpleLinearRegressionScratch()
    model.fit(x_exp, y_exp)
    beta1_values.append(model.beta_1)
    r2_values.append(model.score(x_test, y_test))

# Plot both experiments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Learning curve
axes[0].plot([int(s * len(x_train)) for s in train_sizes], train_r2_scores,
             label='Train R²', color='blue', linewidth=2)
axes[0].plot([int(s * len(x_train)) for s in train_sizes], test_r2_scores,
             label='Test R²', color='orange', linewidth=2)
axes[0].set_title('Learning Curve: Training Size vs R²', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Training Samples', fontsize=11)
axes[0].set_ylabel('R² Score', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Outlier sensitivity
axes[1].plot(outlier_counts, beta1_values, color='red', marker='o', linewidth=2, markersize=8)
axes[1].set_title('Outlier Sensitivity: β₁ Shift with Outliers', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Extreme Outliers Injected', fontsize=11)
axes[1].set_ylabel('Learned β₁ (slope)', fontsize=11)
axes[1].grid(True, alpha=0.3)
for i, (x_val, y_val) in enumerate(zip(outlier_counts, beta1_values)):
    axes[1].annotate(f'{y_val:.1f}', (x_val, y_val), textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=9)

plt.suptitle('SLR Experiments: Learning Curve & Outlier Sensitivity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"β₁ with 0 outliers  : {beta1_values[0]:.2f}")
print(f"β₁ with 20 outliers : {beta1_values[-1]:.2f}")
print(f"→ Outliers shifted β₁ by {abs(beta1_values[-1] - beta1_values[0]):.2f} — OLS is sensitive to outliers!")

## Part 5: Interview Corner

**Question: How are β₀ and β₁ derived in OLS? Walk me through it.**

This is a fundamental FAANG interview question for ML roles. Here is the complete answer:

**Goal:** Find β₀ and β₁ that minimize the Sum of Squared Residuals (SSR):
```
SSR = Σ(yᵢ - ŷᵢ)² = Σ(yᵢ - β₀ - β₁xᵢ)²
```

**Step 1:** Take partial derivative with respect to β₀, set to zero:
```
∂SSR/∂β₀ = 0  →  β₀ = ȳ - β₁x̄
```

**Step 2:** Take partial derivative with respect to β₁, set to zero:
```
∂SSR/∂β₁ = 0  →  β₁ = Σ(xᵢ - x̄)(yᵢ - ȳ) / Σ(xᵢ - x̄)²
                      = Cov(x, y) / Var(x)
```

**Key insight:** β₀ ensures the line always passes through (x̄, ȳ) — the centroid of the data. β₁ captures how much y changes per unit change in x, normalized by x's own variability.

**Follow-up: What if two features are perfectly correlated?**  
The denominator Σ(xᵢ - x̄)² becomes near-zero → β₁ blows up → numerical instability. This is why Ridge regression adds a stabilizing term (α×I) to the denominator.

## Key Takeaways

Remember these 5 bullets for placement interviews:

- **SLR fits ŷ = β₀ + β₁x using OLS**: β₁ = Cov(x,y)/Var(x) is the slope; β₀ = ȳ - β₁x̄ is the intercept. Two numbers define the entire model — computed in one pass, no iteration.
- **OLS minimizes sum of squared residuals**: The line is positioned so total squared vertical distance from all points to the line is minimized. Squaring penalizes large errors more than small ones.
- **Check LINE assumptions with residual plots**: Residuals should be random, centered at zero, and normally distributed. A fan shape means heteroscedasticity; a curve means non-linearity.
- **SLR is highly sensitive to outliers**: One extreme point can significantly shift β₁. Always visualize data before fitting and consider robust regression if outliers are present.
- **R² alone is not enough**: A line through non-linear data can have high R² but violate every assumption. Always pair R² with residual analysis and domain knowledge.